# AgriNova — Disease Detection Training Notebook
**Stage 2: Rice & Cotton Disease Models**

### Architecture Decision
- Crop Classifier already exists and achieves 100% accuracy (EfficientNetV2B0)
- This notebook trains TWO separate disease models (Rice, Cotton)
- Strategy: Original-only test set for scientific integrity; Original + Augmented for training
- Compares EfficientNetV2B0, EfficientNetV2B3, MobileNetV3Large, ResNet50, DenseNet121
- Saves best model per crop based on validation accuracy

## 0. Environment Setup

In [ ]:
# Install / upgrade if running on fresh Colab / Kaggle
# !pip install tensorflow>=2.16 scikit-learn albumentations opencv-python openpyxl tqdm -q

import os, sys, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.applications import (
    EfficientNetV2B0, EfficientNetV2B3,
    MobileNetV3Large, ResNet50, DenseNet121
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
import openpyxl
warnings.filterwarnings('ignore')

print(f'TensorFlow: {tf.__version__}')
print(f'GPUs available: {tf.config.list_physical_devices("GPU")}')

# Mixed precision for GPU speed
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision: enabled')
except:
    print('Mixed precision: not available, using float32')

## 1. Configuration

In [ ]:
# ============================================================
#  PATHS — Edit these to match your environment
# ============================================================
BASE_DATA_PATH   = "disease_detection/Rice_Cotton_disease_Data"   # original images root
AUG_DATA_PATH    = "disease_detection/AGRINOVA_DATASET"           # offline-augmented images root
MODELS_SAVE_PATH = "disease_detection/models"                     # where .keras files go
REPORTS_PATH     = "reports"                                       # evaluation outputs

# ============================================================
#  Hyperparameters
# ============================================================
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
EPOCHS_STAGE1 = 20    # frozen base — fast feature extraction
EPOCHS_STAGE2 = 15    # unfrozen top layers — fine-tuning
UNFREEZE_LAYERS = 40  # how many top layers to unfreeze in stage 2
SEED = 42

os.makedirs(MODELS_SAVE_PATH, exist_ok=True)
os.makedirs(REPORTS_PATH, exist_ok=True)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Dataset Analysis & Quality Check

In [ ]:
def analyze_dataset(root_path, dataset_name):
    """
    Walks a class-folder dataset and reports:
    - Per-class counts
    - Corrupt / unreadable files
    - Duplicate detection via file-size fingerprint
    - Class imbalance ratio
    """
    import hashlib, cv2
    
    print(f"\n{'='*55}")
    print(f" Dataset Analysis: {dataset_name}")
    print(f"{'='*55}")
    
    if not os.path.exists(root_path):
        print(f"  [SKIP] Path not found: {root_path}")
        return {}
    
    stats = {}
    corrupt, duplicates = [], []
    seen_hashes = {}
    
    for cls_name in sorted(os.listdir(root_path)):
        cls_path = os.path.join(root_path, cls_name)
        if not os.path.isdir(cls_path):
            continue
        
        count = 0
        for root, _, files in os.walk(cls_path):
            for f in files:
                if not f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                fp = os.path.join(root, f)
                count += 1
                
                # Corruption check
                img = cv2.imread(fp)
                if img is None:
                    corrupt.append(fp)
                    continue
                
                # Duplicate check (MD5 of raw bytes)
                with open(fp, 'rb') as fh:
                    h = hashlib.md5(fh.read()).hexdigest()
                if h in seen_hashes:
                    duplicates.append((fp, seen_hashes[h]))
                else:
                    seen_hashes[h] = fp
        
        stats[cls_name] = count
    
    if not stats:
        print("  No class folders found.")
        return stats
    
    total = sum(stats.values())
    max_c = max(stats.values())
    min_c = min(stats.values())
    imbalance_ratio = max_c / max(min_c, 1)
    
    print(f"\n  {'Class':<35} {'Count':>6}  {'%':>5}")
    print(f"  {'-'*50}")
    for cls, cnt in sorted(stats.items(), key=lambda x: -x[1]):
        print(f"  {cls:<35} {cnt:>6}  {cnt/total*100:>4.1f}%")
    print(f"  {'-'*50}")
    print(f"  Total images   : {total}")
    print(f"  Corrupt files  : {len(corrupt)}")
    print(f"  Duplicates     : {len(duplicates)}")
    print(f"  Imbalance ratio: {imbalance_ratio:.2f}x  (max/min class)")
    
    if imbalance_ratio > 3:
        print("  ⚠️  High imbalance — class weights will be applied during training")
    
    return stats

# Run analysis on both crop original datasets
rice_orig_path   = os.path.join(BASE_DATA_PATH, "Rice_Disease_Data", "Rice")
cotton_orig_path = os.path.join(BASE_DATA_PATH, "Cotton_Disease_Data", "Cotton")

rice_stats   = analyze_dataset(rice_orig_path,   "Rice Disease (Original)")
cotton_stats = analyze_dataset(cotton_orig_path, "Cotton Disease (Original)")

## 3. Augmentation Strategy Decision

In [ ]:
"""
DECISION: Option C — Combine Original + Pre-generated Augmented Data

Reasoning:
1. The offline augmenter already generates 4 deterministic transforms per image
   (flip, rotate, blur, brightness), giving ~7x the original dataset size.
2. Re-running augmentation online per-epoch is expensive and non-deterministic.
3. Strict split protocol: test set = ORIGINAL images ONLY (no augmented copies),
   which prevents data leakage and gives a scientifically valid benchmark.
4. Training set = 70% of original + 100% of augmented images.

Additional online augmentation is applied during training (mild, via tf.data)
to further regularize without generating new disk files.
"""

rice_aug_path   = os.path.join(AUG_DATA_PATH, "Rice_Augmented")
cotton_aug_path = os.path.join(AUG_DATA_PATH, "Cotton_Augmented")

for name, path in [("Rice Augmented", rice_aug_path), ("Cotton Augmented", cotton_aug_path)]:
    if os.path.exists(path):
        n = sum(len(f) for _, _, f in os.walk(path))
        print(f"{name}: {n} augmented files found")
    else:
        print(f"{name}: NOT FOUND at {path}")
        print("  → Run augmenter_offline.py first, OR set USE_AUG=False below")

USE_AUG = True  # Set False to train on original data only (diagnostic mode)

## 4. Preprocessing Pipeline

In [ ]:
def collect_image_paths(original_root, aug_root, split_ratio=0.3, seed=42, use_aug=True):
    """
    Returns:
        train_paths, train_labels, val_paths, val_labels,
        test_paths, test_labels, class_names, class_weights_dict
    
    Split logic:
        - Original images → 70% train_orig / 15% val / 15% test  (stratified, seeded)
        - Augmented images → 100% added to train only
    """
    if not os.path.exists(original_root):
        raise FileNotFoundError(f"Original data not found: {original_root}")
    
    class_names = sorted([
        d for d in os.listdir(original_root)
        if os.path.isdir(os.path.join(original_root, d))
    ])
    class_to_id = {c: i for i, c in enumerate(class_names)}
    
    train_paths, train_labels = [], []
    val_paths,   val_labels   = [], []
    test_paths,  test_labels  = [], []
    
    rng = np.random.RandomState(seed)
    
    for cls_name in class_names:
        cls_path = os.path.join(original_root, cls_name)
        imgs = []
        for root, _, files in os.walk(cls_path):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    imgs.append(os.path.join(root, f))
        
        rng.shuffle(imgs)
        lbl = class_to_id[cls_name]
        n = len(imgs)
        
        # 70 / 15 / 15 split
        n_train = int(n * 0.70)
        n_val   = int(n * 0.15)
        
        train_paths  += imgs[:n_train];            train_labels  += [lbl] * n_train
        val_paths    += imgs[n_train:n_train+n_val]; val_labels  += [lbl] * n_val
        test_paths   += imgs[n_train+n_val:];      test_labels   += [lbl] * (n - n_train - n_val)
    
    # Add augmented images to train only
    if use_aug and aug_root and os.path.exists(aug_root):
        for cls_name in class_names:
            aug_cls = os.path.join(aug_root, cls_name)
            if not os.path.exists(aug_cls):
                continue
            lbl = class_to_id[cls_name]
            for root, _, files in os.walk(aug_cls):
                for f in files:
                    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                        train_paths.append(os.path.join(root, f))
                        train_labels.append(lbl)
    
    # Compute class weights for imbalanced data
    unique_classes = np.unique(train_labels)
    cw = compute_class_weight('balanced', classes=unique_classes, y=train_labels)
    class_weights_dict = dict(zip(unique_classes.tolist(), cw.tolist()))
    
    print(f"  Classes  : {class_names}")
    print(f"  Train    : {len(train_paths)} images")
    print(f"  Val      : {len(val_paths)} images")
    print(f"  Test     : {len(test_paths)} images (originals only)")
    
    return (train_paths, train_labels, val_paths, val_labels,
            test_paths, test_labels, class_names, class_weights_dict)


def build_tf_dataset(paths, labels, image_size, batch_size,
                     shuffle=False, augment=False, cache=False):
    """
    Builds a tf.data.Dataset with:
    - Decode + resize
    - EfficientNet-style preprocessing (scale to [0,1], then model normalises internally)
    - Optional online augmentation (random flip, brightness, contrast, rotation)
    """
    AUTOTUNE = tf.data.AUTOTUNE
    
    def decode_img(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, image_size)
        img   = tf.cast(img, tf.float32) / 255.0   # [0,1]
        return img, label
    
    def online_aug(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, 0.15)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        img = tf.clip_by_value(img, 0.0, 1.0)
        return img, label
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_img, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache()
    if augment:
        ds = ds.map(online_aug, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

print("Preprocessing functions defined.")

## 5. Model Architecture Comparison

In [ ]:
def build_transfer_model(backbone_name, num_classes, image_size=(224, 224),
                         dropout=0.5, l2_reg=1e-4, dense_units=512):
    """
    Generic transfer-learning model builder.
    All backbones frozen initially (Stage 1).
    Fine-tuning is handled by unfreeze_top() below.
    """
    input_shape = (*image_size, 3)
    inputs      = layers.Input(shape=input_shape)
    
    backbone_map = {
        'EfficientNetV2B0': EfficientNetV2B0,
        'EfficientNetV2B3': EfficientNetV2B3,
        'MobileNetV3Large': MobileNetV3Large,
        'ResNet50':         ResNet50,
        'DenseNet121':      DenseNet121,
    }
    
    if backbone_name not in backbone_map:
        raise ValueError(f"Unknown backbone: {backbone_name}")
    
    # Preprocess per-backbone (ResNet/DenseNet expect [-1,1], EfficientNet expects [0,255])
    needs_rescale = backbone_name.startswith('EfficientNet')
    if needs_rescale:
        # EfficientNet V2 includes its own preprocessing layer internally
        x_in = layers.Rescaling(255.0)(inputs)  # un-scale from [0,1] → [0,255]
    else:
        # ResNet50 / DenseNet121 use [-1,1] preprocessing
        x_in = layers.Rescaling(2.0, offset=-1.0)(inputs)
    
    base = backbone_map[backbone_name](
        weights='imagenet', include_top=False, input_tensor=x_in
    )
    base.trainable = False  # frozen for Stage 1
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(dense_units, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(dropout * 0.5)(x)
    
    # Output layer — float32 for numerical stability with mixed precision
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    
    model = models.Model(inputs, outputs, name=backbone_name)
    return model, base


def unfreeze_top(base_model, n_layers, lr=1e-5):
    """Unfreeze the top-n layers of base model for fine-tuning (Stage 2)."""
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    # Recompile is handled by the caller


def get_callbacks(model_save_path, patience_es=8, patience_lr=4):
    """Production-grade callback set."""
    return [
        callbacks.ModelCheckpoint(
            filepath=model_save_path,
            monitor='val_accuracy',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=patience_es,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.3,
            patience=patience_lr,
            min_lr=1e-7,
            verbose=1
        )
    ]

print("Model builder functions defined.")

## 6. Evaluation Utilities

In [ ]:
def evaluate_model(model, test_ds, class_names, model_label, crop_name, history_s1, history_s2=None):
    """
    Full evaluation suite:
    - Accuracy, Precision, Recall, F1, per-class classification report
    - Confusion matrix plot
    - Training/validation curves
    - ROC-AUC (OvR)
    Returns a metrics dict.
    """
    y_true, y_pred, y_prob = [], [], []
    for imgs, lbls in test_ds:
        probs = model.predict(imgs, verbose=0)
        y_true.extend(lbls.numpy().tolist())
        y_pred.extend(np.argmax(probs, axis=1).tolist())
        y_prob.extend(probs.tolist())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    
    acc  = float(np.mean(y_true == y_pred))
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    # ROC-AUC (macro OvR)
    try:
        y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
        roc = roc_auc_score(y_bin, y_prob, multi_class='ovr', average='macro')
    except Exception:
        roc = None
    
    print(f"\n{'='*55}")
    print(f" Results: {model_label} — {crop_name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    if roc: print(f"  ROC-AUC   : {roc:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
    
    # ── Confusion Matrix ──────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f'{crop_name} — {model_label}\nConfusion Matrix (counts)', fontsize=12)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    plt.setp(axes[0].get_xticklabels(), rotation=40, ha='right', fontsize=9)
    plt.setp(axes[0].get_yticklabels(), rotation=0, fontsize=9)
    
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f'{crop_name} — {model_label}\nConfusion Matrix (normalised)', fontsize=12)
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    plt.setp(axes[1].get_xticklabels(), rotation=40, ha='right', fontsize=9)
    plt.setp(axes[1].get_yticklabels(), rotation=0, fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_PATH, f"{crop_name}_{model_label}_cm.png"), dpi=150)
    plt.show()
    
    # ── Training Curves ───────────────────────────────────────
    def combine_history(h1, h2):
        """Merge Stage 1 + Stage 2 history dicts."""
        if h2 is None:
            return h1.history
        merged = {}
        for k in h1.history:
            merged[k] = h1.history[k] + h2.history.get(k, [])
        return merged
    
    hist = combine_history(history_s1, history_s2)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(hist['accuracy'],     label='Train Acc')
    axes[0].plot(hist['val_accuracy'], label='Val Acc')
    axes[0].set_title(f'{crop_name} — {model_label}\nAccuracy Curve')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(hist['loss'],     label='Train Loss')
    axes[1].plot(hist['val_loss'], label='Val Loss')
    axes[1].set_title(f'{crop_name} — {model_label}\nLoss Curve')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_PATH, f"{crop_name}_{model_label}_curves.png"), dpi=150)
    plt.show()
    
    return {
        'Model':             model_label,
        'Dataset':           crop_name,
        'Accuracy':          round(acc, 4),
        'Precision':         round(prec, 4),
        'Recall':            round(rec, 4),
        'F1_Score':          round(f1, 4),
        'ROC_AUC':           round(roc, 4) if roc else None,
        'Val_Accuracy':      round(max(hist.get('val_accuracy', [0])), 4),
        'Val_Loss':          round(min(hist.get('val_loss', [999])), 4),
    }

print("Evaluation functions defined.")

## 7. Training Pipeline — Single Model Run

In [ ]:
def train_one_model(backbone_name, crop_name, orig_root, aug_root, use_aug=True):
    """
    Full two-stage training for one backbone + crop combo.
    Stage 1: Frozen base, warm-up (20 epochs)
    Stage 2: Unfreeze top-40 layers, fine-tune at low LR (15 epochs)
    Returns metrics dict + elapsed time.
    """
    print(f"\n{'#'*60}")
    print(f"  Training: {backbone_name} | {crop_name}")
    print(f"{'#'*60}")
    
    t0 = time.time()
    
    # ── Data ──────────────────────────────────────────────────
    (train_p, train_l, val_p, val_l,
     test_p, test_l, class_names, cw) = collect_image_paths(
        orig_root, aug_root if use_aug else None, seed=SEED, use_aug=use_aug
    )
    
    train_ds = build_tf_dataset(train_p, train_l, IMAGE_SIZE, BATCH_SIZE,
                                shuffle=True, augment=True)
    val_ds   = build_tf_dataset(val_p,   val_l,   IMAGE_SIZE, BATCH_SIZE)
    test_ds  = build_tf_dataset(test_p,  test_l,  IMAGE_SIZE, BATCH_SIZE)
    
    num_classes = len(class_names)
    
    # ── Build ─────────────────────────────────────────────────
    model, base = build_transfer_model(backbone_name, num_classes)
    
    # ── Stage 1: Warm-up (frozen base) ────────────────────────
    ckpt_path = os.path.join(MODELS_SAVE_PATH, f"_temp_{crop_name}_{backbone_name}.keras")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f"\n[Stage 1] Frozen base warm-up ({EPOCHS_STAGE1} epochs max)")
    history1 = model.fit(
        train_ds,
        epochs=EPOCHS_STAGE1,
        validation_data=val_ds,
        class_weight=cw,
        callbacks=get_callbacks(ckpt_path, patience_es=6, patience_lr=3),
        verbose=1
    )
    
    # ── Stage 2: Fine-tuning (unfreeze top layers) ─────────────
    unfreeze_top(base, UNFREEZE_LAYERS)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(5e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f"\n[Stage 2] Fine-tuning top-{UNFREEZE_LAYERS} layers ({EPOCHS_STAGE2} epochs max)")
    history2 = model.fit(
        train_ds,
        epochs=EPOCHS_STAGE2,
        validation_data=val_ds,
        class_weight=cw,
        callbacks=get_callbacks(ckpt_path, patience_es=5, patience_lr=3),
        verbose=1
    )
    
    elapsed = time.time() - t0
    
    # ── Evaluate ──────────────────────────────────────────────
    metrics = evaluate_model(model, test_ds, class_names,
                             backbone_name, crop_name, history1, history2)
    metrics['Training_Time_s'] = round(elapsed, 1)
    
    return model, class_names, metrics

print("Training pipeline defined.")

## 8. Architecture Comparison (Rice)

In [ ]:
# Architectures to compare
BACKBONES = [
    'EfficientNetV2B0',
    'EfficientNetV2B3',
    'MobileNetV3Large',
    'ResNet50',
    'DenseNet121',
]

rice_results   = []
rice_models    = {}
rice_classes   = None

for bb in BACKBONES:
    model, cls_names, metrics = train_one_model(
        backbone_name=bb,
        crop_name='Rice',
        orig_root=rice_orig_path,
        aug_root=rice_aug_path,
        use_aug=USE_AUG
    )
    rice_results.append(metrics)
    rice_models[bb] = model
    rice_classes = cls_names
    tf.keras.backend.clear_session()   # free GPU memory between runs

rice_df = pd.DataFrame(rice_results).sort_values('Val_Accuracy', ascending=False)
print("\nRice Architecture Comparison:")
print(rice_df[['Model','Accuracy','F1_Score','Val_Accuracy','Training_Time_s']].to_string(index=False))

## 9. Architecture Comparison (Cotton)

In [ ]:
cotton_results  = []
cotton_models   = {}
cotton_classes  = None

for bb in BACKBONES:
    model, cls_names, metrics = train_one_model(
        backbone_name=bb,
        crop_name='Cotton',
        orig_root=cotton_orig_path,
        aug_root=cotton_aug_path,
        use_aug=USE_AUG
    )
    cotton_results.append(metrics)
    cotton_models[bb] = model
    cotton_classes = cls_names
    tf.keras.backend.clear_session()

cotton_df = pd.DataFrame(cotton_results).sort_values('Val_Accuracy', ascending=False)
print("\nCotton Architecture Comparison:")
print(cotton_df[['Model','Accuracy','F1_Score','Val_Accuracy','Training_Time_s']].to_string(index=False))

## 10. Select & Save Best Models

In [ ]:
# ── Select best backbone per crop ─────────────────────────────
best_rice_row    = rice_df.iloc[0]
best_cotton_row  = cotton_df.iloc[0]

best_rice_bb    = best_rice_row['Model']
best_cotton_bb  = best_cotton_row['Model']

print(f"Best Rice model    : {best_rice_bb}  (val_acc={best_rice_row['Val_Accuracy']:.4f})")
print(f"Best Cotton model  : {best_cotton_bb}  (val_acc={best_cotton_row['Val_Accuracy']:.4f})")

# ── Retrain best backbone from scratch for final clean save ───
print("\nRetraining best models for final clean save...")

final_rice_model, rice_classes, _ = train_one_model(
    best_rice_bb, 'Rice', rice_orig_path, rice_aug_path, use_aug=USE_AUG)

final_cotton_model, cotton_classes, _ = train_one_model(
    best_cotton_bb, 'Cotton', cotton_orig_path, cotton_aug_path, use_aug=USE_AUG)

# ── Save models + label files ─────────────────────────────────
rice_model_path   = os.path.join(MODELS_SAVE_PATH, 'rice_disease_model.keras')
cotton_model_path = os.path.join(MODELS_SAVE_PATH, 'cotton_disease_model.keras')

final_rice_model.save(rice_model_path)
final_cotton_model.save(cotton_model_path)

with open(os.path.join(MODELS_SAVE_PATH, 'rice_model_labels.pkl'), 'wb') as f:
    pickle.dump(rice_classes, f)

with open(os.path.join(MODELS_SAVE_PATH, 'cotton_model_labels.pkl'), 'wb') as f:
    pickle.dump(cotton_classes, f)

print(f"\n[SAVED] {rice_model_path}")
print(f"[SAVED] {cotton_model_path}")

## 11. Save Evaluation Reports (Excel + CSV)

In [ ]:
all_results = rice_results + cotton_results
results_df  = pd.DataFrame(all_results)

# Reorder columns
cols = ['Model','Dataset','Accuracy','Precision','Recall','F1_Score',
        'ROC_AUC','Val_Accuracy','Val_Loss','Training_Time_s']
results_df = results_df[[c for c in cols if c in results_df.columns]]

csv_path   = os.path.join(REPORTS_PATH, 'agrinova_model_comparison.csv')
xlsx_path  = os.path.join(REPORTS_PATH, 'agrinova_model_comparison.xlsx')

results_df.to_csv(csv_path, index=False)

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    results_df.to_excel(writer, sheet_name='All Models', index=False)
    rice_df.to_excel(writer, sheet_name='Rice Comparison', index=False)
    cotton_df.to_excel(writer, sheet_name='Cotton Comparison', index=False)

print(f"[SAVED] {csv_path}")
print(f"[SAVED] {xlsx_path}")
print(results_df.to_string(index=False))

## 12. Quick Inference Test

In [ ]:
def quick_infer(model, label_list, image_path):
    """Sanity-check a single image against a loaded model."""
    import cv2
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMAGE_SIZE)
    img = img.astype('float32') / 255.0
    inp = np.expand_dims(img, axis=0)
    probs = model.predict(inp, verbose=0)[0]
    idx   = np.argmax(probs)
    print(f"Predicted : {label_list[idx]}")
    print(f"Confidence: {probs[idx]*100:.2f}%")
    print("All class probabilities:")
    for name, p in zip(label_list, probs):
        bar = '█' * int(p * 40)
        print(f"  {name:<35} {p*100:5.1f}%  {bar}")

# Example (uncomment and replace path):
# quick_infer(final_rice_model, rice_classes, 'path/to/test/leaf.jpg')
print("Inference utility defined. Uncomment example to test.")